# 第17课：自定义 Tool 与 MCP 接入

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter17_自定义Tool与MCP_课后练习.ipynb](chapter17_自定义Tool与MCP_课后练习.ipynb)。

**阶段定位**：阶段三 · 任务二 2.1+2.2 / 15分。15分

有了 Schema 还不够：模型要能**调用工具**。本课先做带契约的本地 Tool，再演示 MCP 握手与能力发现。任务二 2.1 占 10 分，2.2 占 5 分。

## 学习目标

1. 用 ArgsSchema 与 ResultSchema 定义 Tool 契约。
2. 执行逻辑隔离异常：返回 ok=False 与 error 文本，不中断程序。
3. 导出 Function Calling Schema。
4. MCP Client 握手并列出远程（教学为进程内）Tools 与 Resources。

## 学习知识点

| Tool 10分 | MCP 5分 |
| --- | --- |
| Args / Result | handshake ok |
| export_function_schema | tools 元数据 |
| 校验失败不崩溃 | resources 清单 |

## 基础回顾与案例提问

1. **R.1** sku='x' 长度不足，应 raise 还是返回 ok=False？
2. **R.2** Function Calling 的 parameters 从哪份 Schema 来？
3. **R.3** 握手成功是否等于已经执行了 kb_lookup？

教学 MCP 在进程内模拟 stdio 握手，避免课堂依赖外部 npx 进程。概念上仍区分 Client / Server。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. 契约：Args 与 Result

### 理论知识

**入参、出参都是 BaseModel。** 库存工具：sku 最短 3 字符；结果带 stock 与 ok/error。

### 案例：看库存表


In [ ]:
from agent_lab.tools import STOCK, InventoryArgs, InventoryResult
print(STOCK)
print(InventoryArgs.model_json_schema()["properties"])


### 讲解

WIDGET-X 库存 12，WIDGET-MINI 库存 4。未知 SKU 是业务失败，不是 Python 异常。

### 易错点与练习

1. **K1.1** 为什么 Result 也要 Schema，而不是随便 return dict？
2. **K1.2** min_length=3 拦的是哪类胡言？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 导出 Function Calling Schema

### 理论知识

**给模型看的是 JSON，不是 Python 类。** `export_function_schema` 包一层 type=function。

### 案例：导出


In [ ]:
import json
from agent_lab.tools import inventory_tool
print(json.dumps(inventory_tool.export_function_schema(), ensure_ascii=False, indent=2)[:400])


### 讲解

评分老师会看 name 是否为 inventory_lookup，以及 parameters 是否含 sku。

### 易错点与练习

1. **K2.1** description 空着会有什么后果？
2. **K2.2** 为什么 parameters 用 args_schema.model_json_schema()？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 正常调用

### 理论知识

**run() 先 validate 再执行。**

### 案例：查 WIDGET-X


In [ ]:
good = inventory_tool.run(sku="WIDGET-X")
print(good)


### 讲解

stock 必须是 12。不要在作业里重新发明另一张库存表还声称同一验收。

### 易错点与练习

1. **K3.1** sku 大小写不统一时应否在工具内部归一？本课 lookup 会 upper。
2. **K3.2** run 的返回值类型是 BaseModel 还是 dict？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 参数校验隔离

### 理论知识

**太短的 sku 不得让 Jupyter 停在 Traceback。**

### 案例：sku='x'


In [ ]:
isolated = inventory_tool.run(sku="x")
print(isolated.ok, isolated.error)


### 讲解

error 来自 Pydantic 的第一条 msg。程序继续，Agent 才能把错误喂回模型。

### 易错点与练习

1. **K4.1** 若 Tool.run 直接 raise，Self-Correction 还做不做得到？
2. **K4.2** 空 sku 与未知 sku 是同一类失败吗？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 业务异常隔离

### 理论知识

**未知 SKU 走业务分支。** ok=False，error='未知 SKU'。

### 案例：UNKNOWN-SKU


In [ ]:
unknown = inventory_tool.run(sku="UNKNOWN-SKU")
print(unknown.ok, unknown.error, unknown.stock)


### 讲解

stock=0 且 ok=False。不要只看数字 0，免费商品也可能是 0 库存。

### 易错点与练习

1. **K5.1** 为什么未知 SKU 不直接 ValidationError？
2. **K5.2** 隔离的“标准错误文本”会在第 18 课自愈时用到，指的是哪一字段？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. MCP 握手模型

### 理论知识

**Client 与 Server 先握手，再发现能力。** 传输可以是 stdio 或 SSE。本课教学实现 transport='stdio'。

### 案例：握手


In [ ]:
from agent_lab.mcp import MCPClient
hs = MCPClient(transport="stdio").handshake()
print(hs.ok, hs.protocol, hs.server)


### 讲解

server 名为 lab-mcp。真实项目里这一步还会交换协议版本。

### 易错点与练习

1. **K6.1** stdio 和 SSE 对“谁启动谁”的直觉差别是什么？
2. **K6.2** handshake 返回的是元数据还是工具执行结果？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 能力发现

### 理论知识

**列出 Tools 与 Resources。** 本课要求完整读取清单，不要求真的调 kb_lookup。

### 案例：打印清单


In [ ]:
print([t.name for t in hs.tools])
print([r.uri for r in hs.resources])


### 讲解

应看到 kb_lookup、echo_time 与 kb://manual、kb://policy。5 分就给在这份清单上。

### 易错点与练习

1. **K7.1** Resource 的 mime_type 有什么用？
2. **K7.2** 为什么发现与调用要分成两步？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 15 分评分对照

### 理论知识

**2.1 与 2.2 分开给分。** 只会 MCP 不会写 Tool 拿不到 10 分。

### 案例：自检


In [ ]:
assert good.ok and good.stock == 12
assert isolated.ok is False
assert hs.ok and len(hs.tools) >= 2
print("2.1+2.2 课堂自检通过")


### 讲解

下一课把这些 Tool 收成 Skill，MCP 工具也可以被组合进去。

### 易错点与练习

1. **K8.1** export 缺少 parameters 时 2.1 能否满分？
2. **K8.2** 只握手成功但 tools 为空，2.2 能否满分？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：Tool + MCP

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　导出 Schema 并正常调用

打印 function name 与 properties，run WIDGET-X。


In [ ]:
# P1.1: schema + happy path.


### P1.2　两类隔离

短 sku 与未知 sku 都不得崩溃，打印 ok 与 error。


In [ ]:
# P1.2: isolation.


### P1.3　MCP 清单

handshake 后列出 tools 与 resources。


In [ ]:
# P1.3: MCP metadata.


课后请打开 [chapter17_自定义Tool与MCP_课后练习.ipynb](chapter17_自定义Tool与MCP_课后练习.ipynb)。P1 写清契约，P2 做 MCP 发现，P3 选做一枚新 Tool。
